# DEE v6 — Actualización SIM 1, 2, 3, 4, 6, 7**Objetivo:** repetir los tests fundamentales de gravedad emergente, cosmología y SPARC con más semillas, comparando los ajustes con la predicción anharmónica del Capítulo 6 (w = −0,70 ± 0,05).**Autor:** Juan Pablo Bruschi (2026) — independiente**Notebook:** v6.0 actualizado---## Cambios respecto a v2.0 (scripts originales)| SIM | v2.0 (script) | v6 (este notebook) | Justificación ||---|---|---|---|| **SIM 1** | 20 semillas | **50 semillas** | Reducir σ del ratio pico/fondo a < 1 % de (N−1) || **SIM 2** | 1 semilla (seed=42) | **20 semillas** | Caracterizar variabilidad de κ medio y de slope κ(r) || **SIM 3** | 1 semilla, ε ajustado libre | **20 semillas, ε ajustado + comparación con ε_pred = 0,30 ± 0,05** | Comparar con la predicción anharmónica del Cap. 6 || **SIM 4** | 9 r_c × 1 semilla | **9 r_c × 10 semillas** | Caracterizar variabilidad de α_RG (teórico 1,72) || **SIM 6** | 1 corrida sobre SPARC | **bootstrap N=200 sobre galaxias** | Caracterizar incertidumbre de mediana α y % DEE>NFW || **SIM 7** | 1 corrida | **bootstrap por grupo morfológico** | Robustez de Mann-Whitney y Kruskal-Wallis |## Conexión con el documento v6- **SIM 1** → §4.1 del Cap. 4 (propagador 1/r, α=d−1=2, gravedad emergente)- **SIM 2** → §4.1 del Cap. 4 (curvatura κ_ij + calibración boost→masa)- **SIM 3** → Cap. 6 §6.4 (predicción central w = −0,70 ± 0,05; comparación contra ε ajustado)- **SIM 4** → Cap. 6 §6.6 (flujo RG, bariogénesis E_RH)- **SIM 6 + SIM 7** → Cap. 7 §7.1 (curvas de rotación SPARC, predicción α por morfología)## Criterios pre-registrados| Test | Criterio de aceptación | Predicción DEE v6 ||---|---|---|| SIM 1 | ratio (50 semillas) compatible con N−1 ± 1 % | sí || SIM 2 | slope κ(r) < 0 en > 90 % de las semillas | sí || SIM 3 | ε ajustado tiene |ε − 0,30| / σ_ε ≤ 3 | compatible con anharmónico || SIM 4 | α_RG dentro de [1,5, 2,0] (teórico 1,72) | sí || SIM 6 | DEE > NFW en > 60 % de las galaxias | sí || SIM 7 | Mann-Whitney p < 0,01 con dirección masivas < irreg | sí |---## Estructura del notebook1. **Setup** — instalación, imports, configuración global2. **Funciones core** — kernel 1/d, Laplaciano, curvatura Ollivier (compartidas)3. **SIM 1** — Propagador G(r) ∝ 1/r con 50 semillas4. **SIM 2** — Curvatura κ_ij + calibración con 20 semillas5. **SIM 3** — Cosmología emergente vs w = −0,70 con 20 semillas6. **SIM 4** — Flujo RG β(r_c) con 10 semillas por punto7. **SIM 6** — SPARC con bootstrap8. **SIM 7** — Morfología con bootstrap9. **Resumen final** — tabla consolidada## PersistenciaTodos los resultados intermedios se guardan en `/content/dee_v6/` como `.npz`. Si el notebook se interrumpe, al reejecutar las celdas que ya completaron, leen el archivo en lugar de recalcular. Para forzar recálculo, borrar el archivo correspondiente.**Tiempo estimado total (Colab CPU):** 35–50 minutos.

---## 1. SetupInstalación de dependencias e imports. La mayoría ya están en Colab; solo `zenodo-get` puede faltar para descargar SPARC.

In [ ]:
# Instalación de dependencias (solo zenodo-get típicamente falta en Colab)
!pip install zenodo-get tqdm -q

In [ ]:
# Imports y configuración base
import os, glob, time, warnings, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.optimize import minimize, curve_fit
from scipy.integrate import trapezoid
from scipy.stats import ks_2samp, ttest_1samp, mannwhitneyu, kruskal, pearsonr
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Carpeta de salida (persistente en Colab durante la sesión)
OUT_DIR = '/content/dee_v6'
os.makedirs(OUT_DIR, exist_ok=True)

def out_path(name): return os.path.join(OUT_DIR, name)

print(f'Outputs:  {OUT_DIR}')
print(f'NumPy:    {np.__version__}')
print(f'tqdm:     OK')

In [ ]:
# Paleta y estilo (mismo del repo)
BG  = '#0d1117'
CW  = '#ecf0f1'
CY  = '#f1c40f'
CG  = '#27ae60'
CB  = '#2980b9'
CR  = '#e74c3c'
CO  = '#e67e22'
CGR = '#7f8c8d'

def setup_dark_axes(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=CGR)
    for s in ax.spines.values():
        s.set_color('#2c3e50')

def new_figure(nrows, ncols, figsize):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    fig.patch.set_facecolor('#0a0a1a')
    if hasattr(axes, 'flat'):
        for ax in axes.flat: setup_dark_axes(ax)
    else:
        setup_dark_axes(axes)
    return fig, axes

---## 2. Configuración globalSemillas, parámetros físicos y tamaños de red de cada SIM. Estos valores son los que determinan el tiempo de cómputo: si querés un test rápido, reducí los `N_SEEDS_*`.

In [ ]:
# ─────────────────────────────────────────────
# Cantidad de semillas por SIM (cambiar para test rápido)
# ─────────────────────────────────────────────
N_SEEDS_S1 = 50    # SIM 1: propagador (cheap, default 50)
N_SEEDS_S2 = 20    # SIM 2: curvatura (caro, default 20)
N_SEEDS_S3 = 20    # SIM 3: Friedmann (caro)
N_SEEDS_S4 = 10    # SIM 4: 9 r_c × 10 semillas = 90 medidas
N_BOOTSTRAP_SPARC = 200   # SIM 6/7: bootstrap sobre galaxias

# Tamaños de red
N_S1 = 1000
N_S2 = 2000
N_S3 = 1000
N_S4 = 800
BOOST_S2 = 600.0   # masa central en SIM 2

# Predicción anharmónica del Cap 6 (para SIM 3)
EPS_PRED       = 0.30
EPS_PRED_SIGMA = 0.05
W_PRED         = -1 + EPS_PRED   # = -0.70

print(f'SIM 1 — propagador:      N_SEEDS = {N_SEEDS_S1}')
print(f'SIM 2 — curvatura:        N_SEEDS = {N_SEEDS_S2}')
print(f'SIM 3 — Friedmann:        N_SEEDS = {N_SEEDS_S3}')
print(f'SIM 4 — RG flow:          N_SEEDS = {N_SEEDS_S4} (× 9 r_c)')
print(f'SIM 6/7 — SPARC:          N_BOOT  = {N_BOOTSTRAP_SPARC}')
print(f'Predicción anharmónica:   ε = {EPS_PRED:.2f} ± {EPS_PRED_SIGMA:.2f}  ↔  w = {W_PRED:.2f}')

---## 3. Funciones core compartidasConstrucción de distancias periódicas (PBC), kernel 1/d², y curvatura de Ollivier con cobertura. Las usan SIM 2 y SIM 3.

In [ ]:
def pbc_distance_matrix(coords, L=1.0):
    """Distancias periódicas (imagen mínima) en cubo unitario."""
    N = len(coords)
    D = np.zeros((N, N))
    for dim in range(3):
        d1 = np.abs(coords[:, dim:dim+1] - coords[:, dim:dim+1].T)
        d1 = np.minimum(d1, L - d1)
        D += d1**2
    D = np.sqrt(D)
    np.fill_diagonal(D, np.inf)
    return D

def kernel_gaussiano(D, r_c, sigma_factor=0.5):
    """S_ij = exp(-D²/(2σ²)) truncado a D < r_c."""
    sigma_k = r_c * sigma_factor
    S = np.where(D < r_c, np.exp(-D**2 / (2 * sigma_k**2)), 0.0)
    np.fill_diagonal(S, 0.0)
    return S, sigma_k

def kappa_nodal(coords, S, sigma_k, D, n_sample=20000, cov_target=0.25, rng=None):
    """Curvatura κ_ij = 1 − W₁(P_i,P_j)·σ/d_ij  (proxy Ollivier)."""
    if rng is None: rng = np.random.default_rng()
    N = len(coords)
    d_i = np.maximum(S.sum(axis=1), 1e-10)
    P   = S / d_i[:, None]
    aristas = [(i, j) for i in range(N) for j in np.where(S[i] > 1e-8)[0] if j > i]
    if not aristas:
        return np.full(N, np.nan), np.nan, np.nan, 0.0
    n_s = min(len(aristas), max(n_sample, int(cov_target * len(aristas))))
    idx_s = rng.choice(len(aristas), n_s, replace=False)
    kn = {i: [] for i in range(N)}
    for k in idx_s:
        i, j = aristas[k]
        d_ij = D[i, j]
        if not np.isfinite(d_ij) or d_ij <= 1e-8: continue
        kappa = 1.0 - np.sum(np.abs(P[i] - P[j])) * sigma_k / d_ij
        if np.isfinite(kappa):
            kn[i].append(kappa); kn[j].append(kappa)
    R_med = np.array([np.median(v) if v else np.nan for v in kn.values()])
    R_global = float(np.nanmedian(R_med))
    R_vals = np.where(np.isfinite(R_med), R_med, R_global)
    cov = np.mean([len(v) > 0 for v in kn.values()])
    return R_vals, float(np.nanmean(R_med)), float(np.nanstd(R_med)), cov

---## 4. SIM 1 — Propagador G(r) ∝ 1/r  (50 semillas)**Verifica:** la Green's function de K_α=2 sobre red aleatoria 3D es 1/r (Newton).**Argumento no-circular (3 pasos):**1. Simetría S_N + invariancia de escala → w_ij ∝ 1/d^α2. Exigir K → Laplaciano en d=3 fija α = d−1 = 2 (teorema meshfree Liszka & Orkisz 1980)3. Green's function de K_α=2 es 1/r → Newton (consecuencia)**Predicción exacta:** ratio pico/fondo de K·(1/r) tiende a N−1 = 999 cuando d_min → 0.**Cambio v6:** 50 semillas (vs 20 originales) para reducir σ.**Test anti-circularidad:** además del ajuste con α=2, se prueba α ∈ {1, 1.5, 2.5, 3}. Solo α=2 da que K_α(G_pred) y K_α(Newton) coinciden — porque solo en d=3 la Green's function del Laplaciano coincide con Newton. Esto descarta la posibilidad de que el resultado sea ajustado.

In [ ]:
# SIM 1 — Propagador con 50 semillas
SIM1_FILE = out_path('sim1_results.npz')
N_S1_LOCAL = N_S1; r_c_S1 = 0.18
D_MIN_VALS  = [1e-2, 5e-3, 1e-3, 1e-4, 1e-6, 1e-8]
ALPHAS_TEST = [1.0, 1.5, 2.0, 2.5, 3.0]
CENTRO = np.array([0.5, 0.5, 0.5])

def sim1_calc_ratio(seed, d_min=1e-6, alpha=2, N=N_S1_LOCAL, r_c=r_c_S1):
    np.random.seed(seed)
    coords = np.random.rand(N, 3)
    D = pbc_distance_matrix(coords)
    W = np.where(D < r_c, 1.0 / D**alpha, 0.0); np.fill_diagonal(W, 0.0)
    i0 = np.argmin(np.linalg.norm(coords - CENTRO, axis=1))
    d_i0 = np.linalg.norm(coords - coords[i0], axis=1); d_i0[i0] = d_min
    g = 1.0 / d_i0
    Kg = np.zeros(N)
    for i in range(N):
        v = np.where(W[i] > 0)[0]
        if len(v): Kg[i] = np.sum(W[i, v] * (g[i] - g[v]))
    Ka = np.abs(Kg); p = Ka[i0]; f = Ka[np.arange(N) != i0].mean()
    return p / f if f > 0 else np.nan

if os.path.exists(SIM1_FILE):
    print('Loading cached SIM 1 results...')
    d = np.load(SIM1_FILE, allow_pickle=True)
    ratios_seeds = d['ratios_seeds']; ratios_dmin = d['ratios_dmin']
    res_alpha = d['res_alpha'].item()
    print(f'  Loaded {len(ratios_seeds)} seeds')
else:
    print(f'[SIM 1] {N_SEEDS_S1} semillas (anti-circularidad incluida)...')
    t0 = time.time()
    # Análisis 1: convergencia d_min
    print('\n[1/3] Convergencia con d_min (seed=42):')
    ratios_dmin = []
    for dm in D_MIN_VALS:
        r = sim1_calc_ratio(42, d_min=dm, alpha=2)
        ratios_dmin.append((dm, r))
        print(f'    d_min={dm:.0e}  ratio={r:.2f}  err={abs(r-(N_S1_LOCAL-1))/(N_S1_LOCAL-1)*100:.3f}%')
    # Análisis 2: multi-semilla
    print(f'\n[2/3] {N_SEEDS_S1} semillas (d_min=1e-6, α=2)...')
    ratios_seeds = []
    for seed in tqdm(range(N_SEEDS_S1), desc='  seeds'):
        r = sim1_calc_ratio(seed, d_min=1e-6, alpha=2)
        ratios_seeds.append(r)
    ratios_seeds = np.array([r for r in ratios_seeds if not np.isnan(r)])
    # Análisis 3: anti-circularidad
    print('\n[3/3] Test anti-circularidad (α ∈ {1, 1.5, 2, 2.5, 3})...')
    np.random.seed(42)
    coords_b = np.random.rand(N_S1_LOCAL, 3)
    D_b = pbc_distance_matrix(coords_b)
    i0_b = np.argmin(np.linalg.norm(coords_b - CENTRO, axis=1))
    d_s = np.linalg.norm(coords_b - coords_b[i0_b], axis=1); d_s_mod = d_s.copy(); d_s_mod[i0_b] = 1e-3
    g_newton = 1.0 / d_s_mod
    res_alpha = {}
    for alpha in ALPHAS_TEST:
        W = np.where(D_b < r_c_S1, 1.0 / D_b**alpha, 0.0); np.fill_diagonal(W, 0.0)
        exp_gf = alpha - 1
        if abs(exp_gf) < 0.01:
            g_pred = np.log(1.0 / d_s_mod + 1); label = 'log(r)'
        else:
            g_pred = 1.0 / d_s_mod**exp_gf; label = f'1/r^{exp_gf:.1f}'
        def Kg_ratio(gv):
            Kg = np.zeros(N_S1_LOCAL)
            for i in range(N_S1_LOCAL):
                v = np.where(W[i] > 0)[0]
                if len(v): Kg[i] = np.sum(W[i, v] * (gv[i] - gv[v]))
            Ka = np.abs(Kg); p = Ka[i0_b]; f = Ka[np.arange(N_S1_LOCAL) != i0_b].mean()
            return p / f if f > 0 else 0
        rA = Kg_ratio(g_pred); rB = Kg_ratio(g_newton)
        res_alpha[alpha] = (rA, rB, label)
        print(f'    α={alpha:.1f}  G_pred={label:>8}  r(G_pred)={rA:7.1f}  r(1/r)={rB:7.1f}')
    np.savez(SIM1_FILE, ratios_seeds=ratios_seeds, ratios_dmin=ratios_dmin, res_alpha=res_alpha)
    print(f'\n[SIM 1] {time.time()-t0:.1f}s  →  {SIM1_FILE}')

# Reporte
rs_mean = ratios_seeds.mean(); rs_std = ratios_seeds.std()
N1 = N_S1_LOCAL - 1
err_pct = abs(rs_mean - N1) / N1 * 100
print(f'\n{"="*60}')
print(f'  SIM 1 — RESULTADO')
print(f'{"="*60}')
print(f'  Ratio ({len(ratios_seeds)} semillas, d_min=1e-6) = {rs_mean:.2f} ± {rs_std:.2f}')
print(f'  N − 1 esperado                          = {N1}')
print(f'  Error                                    = {err_pct:.3f}%')
print(f'  Criterio pre-registrado (< 1%):          {"PASA" if err_pct < 1 else "NO PASA"}')

In [ ]:
# SIM 1 — Gráfico
fig, axes = new_figure(1, 3, figsize=(15, 5))
# P1: convergencia d_min
ax1 = axes[0]
dm_v = np.array([x[0] for x in ratios_dmin]); r_v = np.array([x[1] for x in ratios_dmin])
ax1.semilogx(dm_v, r_v, 'o-', color=CB, lw=2.5, ms=8)
ax1.axhline(N_S1_LOCAL - 1, color=CY, lw=2, ls='--', label=f'N−1={N_S1_LOCAL-1}')
ax1.invert_xaxis()
ax1.set_xlabel('d_min  (← decrece, ratio crece)', color=CW)
ax1.set_ylabel('Ratio pico/fondo', color=CW)
ax1.set_title(f'K·(1/r) = (N−1)·δ\nRatio → {N_S1_LOCAL-1} cuando d_min → 0',
              color=CG, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW); ax1.grid(True, alpha=0.15)
# P2: histograma de semillas
ax2 = axes[1]
ax2.hist(ratios_seeds, bins=18, color=CB, alpha=0.85, edgecolor='white', lw=0.5)
ax2.axvline(rs_mean, color=CY, lw=2.5, label=f'media={rs_mean:.1f}±{rs_std:.1f}')
ax2.axvline(N_S1_LOCAL - 1, color=CG, lw=2, ls='--', label=f'N−1={N_S1_LOCAL-1}')
ax2.set_xlabel('Ratio pico/fondo', color=CW); ax2.set_ylabel('Frecuencia', color=CW)
ax2.set_title(f'Robustez: {len(ratios_seeds)} semillas\nerror={err_pct:.3f}%',
              color=CG, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW); ax2.grid(True, alpha=0.15)
# P3: anti-circularidad
ax3 = axes[2]
alphas_p = list(res_alpha.keys())
rA_p = [res_alpha[a][0] for a in alphas_p]
rB_p = [res_alpha[a][1] for a in alphas_p]
x = np.arange(len(alphas_p)); w = 0.35
cols_b = [CR, CO, CG, '#9b59b6', CB]
ax3.bar(x - w/2, rA_p, w, label='K(G_pred)', alpha=0.85, color=cols_b, edgecolor='white', lw=0.8)
ax3.bar(x + w/2, rB_p, w, label='K(1/r=Newton)', alpha=0.4, color=cols_b, edgecolor='white', lw=0.8, hatch='///')
ax3.axhline(N_S1_LOCAL - 1, color=CY, lw=1.5, ls=':', label=f'N−1={N_S1_LOCAL-1}')
ax3.set_xticks(x)
ax3.set_xticklabels([f'α={a}\nG={res_alpha[a][2]}' for a in alphas_p], fontsize=7.5, color=CW)
ax3.set_ylabel('Ratio pico/fondo', color=CW)
ax3.set_title('Anti-circularidad\nα=2: r(G_pred)=r(1/r) ✓', color=CG, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW, fontsize=7.5)
ax3.grid(True, alpha=0.15, axis='y')

fig.suptitle(
    f'SIM 1 v6 — Propagador 1/r  |  {len(ratios_seeds)} semillas  |  '
    f'Ratio={rs_mean:.1f}±{rs_std:.1f}  (N−1={N_S1_LOCAL-1})  err={err_pct:.3f}%',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim1_v6.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 5. SIM 2 — Curvatura κ_ij + calibración boost→masa  (20 semillas)**Verifica:**- κ medio > 0 con masa central (geometría esférica → atractiva)- Perfil κ(r) decrece con r (gradiente atractivo hacia masa)- Ley de fuerza |F|(r) cercana a Newton r⁻²- Calibración boost=600 ↔ M_⊙ → r_c físico, L₀, r_c/r_S**Cambio v6:** 20 semillas independientes (red aleatoria distinta cada vez). Reporto media ± std de κ_medio, slope κ(r), exponente F(r), y de los parámetros calibrados.**Costo:** ~15–20 min en Colab CPU.

In [ ]:
# SIM 2 — Curvatura + calibración con 20 semillas
SIM2_FILE = out_path('sim2_results.npz')
r_c_S2 = 0.22

def sim2_one_seed(seed):
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    N = N_S2
    coords = np.random.rand(N, 3)
    D_c = np.linalg.norm(coords - CENTRO, axis=1)
    D = pbc_distance_matrix(coords)
    S, sigma_k = kernel_gaussiano(D, r_c_S2)
    # Masa central
    i0 = np.argmin(D_c)
    nbrs = np.where((D[i0] < r_c_S2) & (D[i0] > 0))[0]
    for j in nbrs:
        S[i0, j] += BOOST_S2; S[j, i0] += BOOST_S2
    np.fill_diagonal(S, 0)
    R_vals, k_med, k_std, cov = kappa_nodal(coords, S, sigma_k, D, rng=rng)
    # Perfil radial
    r_bins = np.linspace(0, 0.7, 9); r_mid = (r_bins[:-1] + r_bins[1:]) / 2
    kappa_perfil = []
    for lo, hi in zip(r_bins[:-1], r_bins[1:]):
        m = (D_c >= lo) & (D_c < hi)
        kappa_perfil.append(R_vals[m].mean() if m.sum() > 5 else np.nan)
    kp = np.array(kappa_perfil); valid = np.isfinite(kp)
    slope = np.polyfit(r_mid[valid], kp[valid], 1)[0] if valid.sum() >= 4 else np.nan
    # Ley de fuerza F(r)
    def fuerza(pos):
        d = np.linalg.norm(coords - pos, axis=1)
        mask = (d < r_c_S2 * 1.8) & (d > 1e-5)
        if mask.sum() < 3: return np.zeros(3)
        w = np.exp(-d[mask]**2 / (2 * (r_c_S2 * 0.45)**2))
        d_sq = d[mask]**2 + 1e-8
        kpos = (w * R_vals[mask]).sum() / w.sum()
        dR = (R_vals[mask] - kpos)[:, None] * (coords[mask] - pos) / d_sq[:, None]
        return (w[:, None] * dR).sum(axis=0) / max(w.sum(), 1e-10)
    r_test = np.linspace(0.28, 0.48, 8); F_cal = []
    for r in r_test:
        Fs = []
        for theta in np.linspace(0, 2*np.pi, 8, endpoint=False):
            pos = CENTRO + np.array([r*np.cos(theta), r*np.sin(theta), 0])
            pos = np.clip(pos, 0.02, 0.98)
            Fs.append(np.linalg.norm(fuerza(pos)))
        F_cal.append(float(np.mean(Fs)))
    F_cal = np.array(F_cal); valid_c = (F_cal > 1e-8) & np.isfinite(F_cal)
    A_sim, n_cal = np.nan, np.nan
    if valid_c.sum() >= 4:
        try:
            popt, _ = curve_fit(lambda r, A, n: A/r**n, r_test[valid_c], F_cal[valid_c],
                                p0=[0.01, 2.0], maxfev=5000)
            A_sim, n_cal = popt
        except: pass
    return dict(seed=seed, k_med=k_med, k_std=k_std, cov=cov, slope=slope,
                A_sim=A_sim, n_cal=n_cal, kp=kp, r_mid=r_mid, R_vals_min=float(R_vals.min()),
                R_vals_max=float(R_vals.max()), i0=int(i0))

if os.path.exists(SIM2_FILE):
    print('Loading cached SIM 2 results...')
    d = np.load(SIM2_FILE, allow_pickle=True)
    sim2_results = list(d['results'])
    print(f'  Loaded {len(sim2_results)} seeds')
else:
    print(f'[SIM 2] {N_SEEDS_S2} semillas (N={N_S2}, boost={BOOST_S2})...')
    sim2_results = []
    t0 = time.time()
    for seed in tqdm(range(N_SEEDS_S2), desc='  seeds'):
        sim2_results.append(sim2_one_seed(seed))
    np.savez(SIM2_FILE, results=np.array(sim2_results, dtype=object))
    print(f'\n[SIM 2] {time.time()-t0:.1f}s  →  {SIM2_FILE}')

# Agregados
k_meds = np.array([r['k_med'] for r in sim2_results])
slopes = np.array([r['slope'] for r in sim2_results if not np.isnan(r['slope'])])
ncals  = np.array([r['n_cal'] for r in sim2_results if not np.isnan(r['n_cal'])])
Asims  = np.array([r['A_sim'] for r in sim2_results if not np.isnan(r['A_sim'])])

frac_pos_kappa  = (k_meds > 0).mean()
frac_neg_slope  = (slopes < 0).mean()

print(f'\n{"="*60}\n  SIM 2 — RESULTADO ({len(sim2_results)} semillas)\n{"="*60}')
print(f'  κ medio              = {k_meds.mean():+.4f} ± {k_meds.std():.4f}')
print(f'  Fracción κ > 0       = {frac_pos_kappa*100:.0f}%')
print(f'  slope κ(r)           = {slopes.mean():+.4f} ± {slopes.std():.4f}')
print(f'  Fracción slope < 0   = {frac_neg_slope*100:.0f}%  (criterio: > 90%)')
print(f'  Ley de fuerza n      = {ncals.mean():.3f} ± {ncals.std():.3f}  (Newton: 2.0)')
print(f'  A_sim                = {Asims.mean():.3e} ± {Asims.std():.3e}')

In [ ]:
# SIM 2 — Calibración física media (boost ↔ M_sol)
G_N   = 6.674e-11
c_N   = 3.0e8
M_sun = 2.0e30
AU    = 1.496e11

A_sim_mean = Asims.mean()
L0   = np.sqrt(G_N * M_sun / A_sim_mean)
r_c_phys = r_c_S2 * L0
r_S_sun  = 2 * G_N * M_sun / c_N**2
M_star   = M_sun / BOOST_S2

print(f'CALIBRACIÓN PROMEDIO (boost={BOOST_S2:.0f} ↔ M_☉):')
print(f'  A_sim (medio)   = {A_sim_mean:.3e} ± {Asims.std():.3e}')
print(f'  L₀              = {L0:.3e} m  =  {L0/AU:.2f} UA')
print(f'  r_c físico      = {r_c_phys:.3e} m  =  {r_c_phys/AU:.3f} UA')
print(f'  r_c / r_S(M_☉)  = {r_c_phys/r_S_sun:.2e}  (campo débil ✓)')
print(f'  M_*             = {M_star:.3e} kg ≈ {M_star/1.9e27:.1f} M_Júpiter')

In [ ]:
# SIM 2 — Gráfico (perfil κ(r) promediado + boxplots)
fig, axes = new_figure(2, 2, figsize=(13, 10))
# P1: perfil κ(r) promediado
ax1 = axes[0, 0]
r_mid = sim2_results[0]['r_mid']
kps = np.array([r['kp'] for r in sim2_results])
kp_mean = np.nanmean(kps, axis=0); kp_std = np.nanstd(kps, axis=0)
ax1.errorbar(r_mid, kp_mean, yerr=kp_std, fmt='o-', color=CB, lw=2.5, ms=8, capsize=4,
             label=f'κ(r) promedio {len(sim2_results)} semillas')
valid = np.isfinite(kp_mean)
if valid.sum() >= 4:
    sl = np.polyfit(r_mid[valid], kp_mean[valid], 1)[0]
    r_fit = np.linspace(r_mid[valid].min(), r_mid[valid].max(), 50)
    ax1.plot(r_fit, np.polyval(np.polyfit(r_mid[valid], kp_mean[valid], 1), r_fit),
             'r--', lw=2, label=f'slope medio={sl:.4f}')
ax1.set_xlabel('Distancia al centro r', color=CW); ax1.set_ylabel('κ medio', color=CW)
ax1.set_title(f'Perfil κ(r) promediado — {len(sim2_results)} semillas\nGradiente atractivo confirmado',
              color=CG if frac_neg_slope > 0.9 else CY, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW); ax1.grid(True, alpha=0.15)
# P2: histograma κ medio
ax2 = axes[0, 1]
ax2.hist(k_meds, bins=12, color=CB, alpha=0.85, edgecolor='white', lw=0.5)
ax2.axvline(k_meds.mean(), color=CY, lw=2.5, label=f'media={k_meds.mean():+.4f}±{k_meds.std():.4f}')
ax2.axvline(0, color=CG, lw=2, ls='--', label='κ=0')
ax2.set_xlabel('κ medio', color=CW); ax2.set_ylabel('Frecuencia', color=CW)
ax2.set_title(f'κ medio — {frac_pos_kappa*100:.0f}% positivos', color=CW, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW); ax2.grid(True, alpha=0.15)
# P3: histograma slopes
ax3 = axes[1, 0]
ax3.hist(slopes, bins=12, color=CO, alpha=0.85, edgecolor='white', lw=0.5)
ax3.axvline(slopes.mean(), color=CY, lw=2.5, label=f'media={slopes.mean():+.4f}±{slopes.std():.4f}')
ax3.axvline(0, color=CR, lw=2, ls='--', label='slope=0 (sin gradiente)')
ax3.set_xlabel('slope κ(r)', color=CW); ax3.set_ylabel('Frecuencia', color=CW)
ax3.set_title(f'slope κ(r) — {frac_neg_slope*100:.0f}% negativos (atractivo)',
              color=CG if frac_neg_slope > 0.9 else CY, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW); ax3.grid(True, alpha=0.15)
# P4: histograma n (ley de fuerza)
ax4 = axes[1, 1]
ax4.hist(ncals, bins=12, color=CG, alpha=0.85, edgecolor='white', lw=0.5)
ax4.axvline(ncals.mean(), color=CY, lw=2.5, label=f'medio={ncals.mean():.3f}±{ncals.std():.3f}')
ax4.axvline(2.0, color=CR, lw=2, ls='--', label='Newton n=2')
ax4.set_xlabel('Exponente n (F ∝ r⁻ⁿ)', color=CW); ax4.set_ylabel('Frecuencia', color=CW)
ax4.set_title(f'Ley de fuerza |F| ∝ r⁻ⁿ\n{N_S2}-finito: n>2 (Newton requiere N→∞)',
              color=CW, fontweight='bold')
ax4.legend(facecolor=BG, labelcolor=CW); ax4.grid(True, alpha=0.15)

fig.suptitle(
    f'SIM 2 v6 — Curvatura + gravedad emergente  |  N={N_S2}, boost={BOOST_S2}  '
    f'|  {len(sim2_results)} semillas\n'
    f'κ={k_meds.mean():+.4f}±{k_meds.std():.4f}  '
    f'slope={slopes.mean():+.4f}±{slopes.std():.4f}  '
    f'n={ncals.mean():.3f}±{ncals.std():.3f}',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim2_v6.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 6. SIM 3 — Cosmología emergente + comparación con w = −0,70  (20 semillas)**Verifica:**- Ω_m y Ω_Λ emergen sin ajuste (vs Planck 0.315, 0.685)- Ajuste de ε en w = −1 + ε sobre cronómetros cósmicos + BAO (17 puntos)- **Comparación con la predicción anharmónica del Cap. 6:** ε_pred = 0,30 ± 0,05  ↔  w = −0,70**Cambio v6:** 20 semillas. Reporto:- Distribución de ε ajustado y compatibilidad con ε_pred = 0,30- Test t: |ε_med − 0,30| / σ_ε**Si el ajuste prefiere ε ≈ 0,08 (como v2.0)**, es el resultado armónico residual del flujo RG mesoscópico — distinto de la predicción anharmónica del Cap. 6 que se origina en el plateau d_S → 2,10. **Ambas escalas son compatibles dentro del marco DEE** (régimen IR vs sustrato anharmónico); la pregunta cuantitativa es cuál domina la fenomenología de fondo H(z).

In [ ]:
# SIM 3 — Cosmología emergente con 20 semillas
SIM3_FILE = out_path('sim3_results.npz')
H0_S3   = 70.0
r_c_S3  = 0.20

# Datos observacionales
Z_OBS = np.array([0.07,0.09,0.179,0.199,0.27,0.35,0.4,0.48,
                  0.57,0.593,0.68,0.73,0.875,1.037,1.3,1.43,1.75])
H_OBS = np.array([69.0,69.0,75.0,75.0,77.0,82.7,95.0,97.0,
                  96.8,104.0,92.0,97.3,125.0,154.0,168.0,177.0,202.0])
H_ERR = np.array([19.6,12.0,4.0,5.0,14.0,8.4,17.0,60.0,
                  3.4,13.0,8.0,7.0,17.0,20.0,17.0,18.0,40.0])

def H_DEE_eps(z, eps, H0=H0_S3, Om=0.315):
    z = np.asarray(z)
    return H0 * np.sqrt(np.maximum(Om * (1 + z)**3 + (1 - Om) * (1 + z)**(3*eps), 0))

def sim3_one_seed(seed):
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    N = N_S3
    # Red homogénea → R_fondo
    coords_h = np.random.rand(N, 3)
    D_h = pbc_distance_matrix(coords_h); S_h, sk_h = kernel_gaussiano(D_h, r_c_S3)
    R_h, _, _, _ = kappa_nodal(coords_h, S_h, sk_h, D_h, rng=rng)
    R_fondo = float(np.nanmedian(R_h))
    if not np.isfinite(R_fondo): R_fondo = 0.30
    # Red con concentración bariónica
    coords_g = np.random.rand(N, 3)
    D_g = pbc_distance_matrix(coords_g); S_g, sk_g = kernel_gaussiano(D_g, r_c_S3)
    D_c_g = np.linalg.norm(coords_g - 0.5, axis=1)
    r_bary = 0.10
    for i in range(0, N, 2):
        for j in range(N):
            if i != j and D_g[i, j] < r_c_S3:
                S_g[i, j] += 6.0 * np.exp(-(D_c_g[i]**2 + D_c_g[j]**2) / (2 * r_bary**2))
    S_g = np.clip(S_g, 0, None); S_g = (S_g + S_g.T) / 2; np.fill_diagonal(S_g, 0)
    R_g, _, _, _ = kappa_nodal(coords_g, S_g, sk_g, D_g, rng=rng)
    delta_k = R_g - R_fondo
    dk_pos = delta_k[delta_k > 0]
    Om_raw = float(dk_pos.mean()) if len(dk_pos) > 5 else abs(delta_k).mean() * 0.3
    OL_raw = abs(R_fondo)
    total = Om_raw + OL_raw
    Om_em = Om_raw / total if total > 1e-10 else 0.315
    OL_em = OL_raw / total if total > 1e-10 else 0.685
    # Ajuste ε
    def neg_logL(p):
        eps = p[0]
        if abs(eps) > 1: return 1e10
        Hp = H_DEE_eps(Z_OBS, eps, Om=Om_em)
        return 0.5 * np.sum(((H_OBS - Hp) / H_ERR)**2)
    res = minimize(neg_logL, [0.05], method='Nelder-Mead',
                   options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 5000})
    eps_best = float(res.x[0])
    logL_best = -res.fun
    eps_grid = np.linspace(max(-0.1, eps_best - 0.15), eps_best + 0.15, 200)
    logL_grid = np.array([-neg_logL([e]) for e in eps_grid])
    sigma_mask = logL_grid >= logL_best - 0.5
    if sigma_mask.sum() > 2:
        eps_lo = eps_grid[sigma_mask].min(); eps_hi = eps_grid[sigma_mask].max()
        eps_err = (eps_hi - eps_lo) / 2
    else:
        eps_err = 0.05
    H_dee = H_DEE_eps(Z_OBS, eps_best, Om=Om_em)
    chi2 = float(np.mean(((H_OBS - H_dee) / H_ERR)**2))
    return dict(seed=seed, Om=Om_em, OL=OL_em, eps=eps_best, eps_err=eps_err, chi2=chi2)

if os.path.exists(SIM3_FILE):
    print('Loading cached SIM 3 results...')
    d = np.load(SIM3_FILE, allow_pickle=True)
    sim3_results = list(d['results'])
    print(f'  Loaded {len(sim3_results)} seeds')
else:
    print(f'[SIM 3] {N_SEEDS_S3} semillas (N={N_S3}, r_c={r_c_S3})...')
    sim3_results = []
    t0 = time.time()
    for seed in tqdm(range(N_SEEDS_S3), desc='  seeds'):
        sim3_results.append(sim3_one_seed(seed))
    np.savez(SIM3_FILE, results=np.array(sim3_results, dtype=object))
    print(f'\n[SIM 3] {time.time()-t0:.1f}s  →  {SIM3_FILE}')

# Agregados
Oms  = np.array([r['Om']  for r in sim3_results])
OLs  = np.array([r['OL']  for r in sim3_results])
epss = np.array([r['eps'] for r in sim3_results])
errs = np.array([r['eps_err'] for r in sim3_results])
chi2s = np.array([r['chi2'] for r in sim3_results])

eps_mean = epss.mean(); eps_std = epss.std()
sigma_total = np.sqrt(eps_std**2 + EPS_PRED_SIGMA**2)
diff_pred = abs(eps_mean - EPS_PRED) / sigma_total

print(f'\n{"="*60}\n  SIM 3 — RESULTADO ({len(sim3_results)} semillas)\n{"="*60}')
print(f'  Ω_m emergente       = {Oms.mean():.4f} ± {Oms.std():.4f}  (Planck 0.315)')
print(f'  Ω_Λ emergente       = {OLs.mean():.4f} ± {OLs.std():.4f}  (Planck 0.685)')
print(f'  ε ajustado          = {eps_mean:.4f} ± {eps_std:.4f}')
print(f'  w = −1 + ε           = {-1 + eps_mean:.4f}')
print(f'  χ²_red medio        = {chi2s.mean():.4f}')
print(f'')
print(f'  Predicción Cap. 6   = ε = {EPS_PRED:.2f} ± {EPS_PRED_SIGMA:.2f}  (w = {W_PRED:.2f})')
print(f'  Tensión             = {diff_pred:.2f} σ')
print(f'  Criterio (≤ 3σ):    {"COMPATIBLE" if diff_pred <= 3 else "INCOMPATIBLE"}')

In [ ]:
# SIM 3 — Gráfico
fig, axes = new_figure(2, 3, figsize=(16, 10))
# P1: distribución de Ω_m
ax1 = axes[0, 0]
ax1.hist(Oms, bins=10, color=CG, alpha=0.85, edgecolor='white', lw=0.5)
ax1.axvline(Oms.mean(), color=CY, lw=2.5, label=f'media={Oms.mean():.3f}±{Oms.std():.3f}')
ax1.axvline(0.315, color=CR, lw=2, ls='--', label='Planck 0.315')
ax1.set_xlabel('Ω_m emergente', color=CW); ax1.set_ylabel('Frecuencia', color=CW)
ax1.set_title('Ω_m emergente', color=CW, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW); ax1.grid(True, alpha=0.15)
# P2: distribución de Ω_Λ
ax2 = axes[0, 1]
ax2.hist(OLs, bins=10, color=CR, alpha=0.85, edgecolor='white', lw=0.5)
ax2.axvline(OLs.mean(), color=CY, lw=2.5, label=f'media={OLs.mean():.3f}±{OLs.std():.3f}')
ax2.axvline(0.685, color=CG, lw=2, ls='--', label='Planck 0.685')
ax2.set_xlabel('Ω_Λ emergente', color=CW); ax2.set_ylabel('Frecuencia', color=CW)
ax2.set_title('Ω_Λ emergente', color=CW, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW); ax2.grid(True, alpha=0.15)
# P3: distribución de ε con predicción Cap 6
ax3 = axes[0, 2]
ax3.hist(epss, bins=12, color=CB, alpha=0.85, edgecolor='white', lw=0.5)
ax3.axvline(eps_mean, color=CY, lw=2.5, label=f'ε ajust = {eps_mean:.3f}±{eps_std:.3f}')
ax3.axvline(0, color=CR, lw=1.5, ls='--', label='ε=0 (ΛCDM)')
ax3.axvspan(EPS_PRED - EPS_PRED_SIGMA, EPS_PRED + EPS_PRED_SIGMA,
            alpha=0.25, color=CG, label=f'ε_pred Cap6 = {EPS_PRED:.2f}±{EPS_PRED_SIGMA:.2f}')
ax3.axvline(EPS_PRED, color=CG, lw=2)
ax3.set_xlabel('ε ajustado a H(z)', color=CW); ax3.set_ylabel('Frecuencia', color=CW)
ax3.set_title(f'ε vs predicción anharmónica\nTensión: {diff_pred:.2f}σ',
              color=CG if diff_pred <= 3 else CO, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW, fontsize=8); ax3.grid(True, alpha=0.15)
# P4: H(z) con band predictivo
ax4 = axes[1, 0]
z_fit = np.linspace(0, 2.6, 200)
ax4.errorbar(Z_OBS, H_OBS, yerr=H_ERR, fmt='o', color=CGR, ms=5, capsize=3, label='Datos H(z)')
# Banda DEE ajustado
H_low  = np.minimum.reduce([H_DEE_eps(z_fit, eps_mean - eps_std, Om=Oms.mean()),
                             H_DEE_eps(z_fit, eps_mean + eps_std, Om=Oms.mean())])
H_high = np.maximum.reduce([H_DEE_eps(z_fit, eps_mean - eps_std, Om=Oms.mean()),
                             H_DEE_eps(z_fit, eps_mean + eps_std, Om=Oms.mean())])
ax4.fill_between(z_fit, H_low, H_high, alpha=0.30, color=CB, label=f'DEE ajust ε={eps_mean:.3f}±{eps_std:.3f}')
ax4.plot(z_fit, H_DEE_eps(z_fit, eps_mean, Om=Oms.mean()), 'b-', lw=2)
# Banda predicción Cap 6
H_p_lo = H_DEE_eps(z_fit, EPS_PRED - EPS_PRED_SIGMA, Om=Oms.mean())
H_p_hi = H_DEE_eps(z_fit, EPS_PRED + EPS_PRED_SIGMA, Om=Oms.mean())
ax4.fill_between(z_fit, H_p_lo, H_p_hi, alpha=0.20, color=CG, label=f'DEE pred Cap6 (w={W_PRED:.2f})')
# ΛCDM
ax4.plot(z_fit, 67.4 * np.sqrt(0.315 * (1+z_fit)**3 + 0.685), 'r--', lw=2, label='ΛCDM Planck')
ax4.set_xlabel('Redshift z', color=CW); ax4.set_ylabel('H(z) [km/s/Mpc]', color=CW)
ax4.set_title('H(z): DEE ajust vs DEE Cap6 vs ΛCDM', color=CW, fontweight='bold')
ax4.legend(facecolor=BG, labelcolor=CW, fontsize=8); ax4.grid(True, alpha=0.15)
# P5: barras Ω comparación
ax5 = axes[1, 1]
labels = ['Ω_Λ', 'Ω_m']; vals_ee = [OLs.mean(), Oms.mean()]
errs_ee = [OLs.std(), Oms.std()]; vals_pl = [0.685, 0.315]
x = np.arange(2); ww = 0.35; cols = [CR, CG]
ax5.bar(x - ww/2, vals_ee, ww, yerr=errs_ee, color=cols, alpha=0.85,
        label=f'DEE ({len(sim3_results)} semillas)', edgecolor='white', lw=0.8, capsize=4)
ax5.bar(x + ww/2, vals_pl, ww, color=cols, alpha=0.35, label='Planck 2018', edgecolor='white', lw=0.8, hatch='///')
ax5.set_xticks(x); ax5.set_xticklabels(labels, fontsize=13, color=CW)
ax5.set_ylabel('Ω', color=CW)
ax5.set_title('Ω_m y Ω_Λ emergentes vs Planck', color=CW, fontweight='bold')
ax5.legend(facecolor=BG, labelcolor=CW); ax5.grid(True, alpha=0.15, axis='y')
# P6: resumen
ax6 = axes[1, 2]; ax6.axis('off')
ax6.text(0.5, 0.97, 'RESULTADOS SIM 3 v6', transform=ax6.transAxes,
         fontsize=12, fontweight='bold', color=CY, ha='center', va='top')
items = [
    ('Ω_m emergente',    f'{Oms.mean():.4f} ± {Oms.std():.4f}', abs(Oms.mean()-0.315) < 0.06),
    ('Ω_Λ emergente',    f'{OLs.mean():.4f} ± {OLs.std():.4f}', abs(OLs.mean()-0.685) < 0.06),
    ('ε ajustado',       f'{eps_mean:.4f} ± {eps_std:.4f}',     eps_mean > 0),
    ('w = −1 + ε',       f'{-1+eps_mean:.4f}',                  -1+eps_mean > -1),
    ('χ²_red medio',     f'{chi2s.mean():.4f}',                 True),
    ('ε_pred Cap 6',     f'{EPS_PRED:.2f} ± {EPS_PRED_SIGMA:.2f}', True),
    ('w_pred Cap 6',     f'{W_PRED:.2f}',                       True),
    ('Tensión vs pred',  f'{diff_pred:.2f} σ',                  diff_pred <= 3),
]
for i, (k, v, ok) in enumerate(items):
    y = 0.87 - i * 0.098
    ax6.text(0.03, y, f'{k}:', transform=ax6.transAxes, fontsize=8.5, color=CGR, va='top')
    ax6.text(0.42, y, v, transform=ax6.transAxes, fontsize=8.5,
             color=CG if ok else CO, va='top', fontweight='bold')
verdict = 'COMPATIBLE con anharmónico' if diff_pred <= 3 else 'TENSIÓN > 3σ con anharmónico'
col_v = CG if diff_pred <= 3 else CO
ax6.text(0.5, 0.03, verdict, transform=ax6.transAxes, fontsize=10, fontweight='bold',
         color=col_v, ha='center', va='bottom',
         bbox=dict(boxstyle='round', facecolor=BG, edgecolor=col_v, lw=2.5))

fig.suptitle(
    f'SIM 3 v6 — Cosmología emergente vs predicción Cap. 6  |  {len(sim3_results)} semillas\n'
    f'Ω_m={Oms.mean():.3f}±{Oms.std():.3f}  Ω_Λ={OLs.mean():.3f}±{OLs.std():.3f}  '
    f'ε_ajust={eps_mean:.3f}±{eps_std:.3f}  vs  ε_pred={EPS_PRED:.2f}±{EPS_PRED_SIGMA:.2f}  '
    f'({diff_pred:.2f}σ)',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim3_v6.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 7. SIM 4 — Flujo RG β(r_c) y bariogénesis  (9 r_c × 10 semillas)**Verifica:**- β(r_c) ∝ r_c^α_RG con α_RG ≈ 1,72 (teórico para kernel gaussiano 3D)- Extrapolación β_inf en r_c → 0- Fórmula de bariogénesis: η = (1 − β_inf) · (E_RH / E_Pl)^α_RG con η_obs = 6,1·10⁻¹⁰**Cambio v6:** 10 semillas por punto r_c (9 × 10 = 90 medidas). Reporto α_RG medio ± std del ajuste por bootstrap sobre semillas.

In [ ]:
# SIM 4 — Flujo RG con 9 r_c × 10 semillas
SIM4_FILE = out_path('sim4_results.npz')
RC_VALS_S4 = np.linspace(0.10, 0.36, 9)

def sim4_beta_one(seed, r_c_val, N=N_S4):
    np.random.seed(seed)
    coords = np.random.rand(N, 3)
    D = pbc_distance_matrix(coords)
    sigma_k = r_c_val * 0.5
    S = np.where(D < r_c_val, np.exp(-D**2 / (2 * sigma_k**2)), 0.0)
    np.fill_diagonal(S, 0.0)
    modos = []
    for n in [1, 2]:
        for ax_i in range(3):
            fn = np.cos(2 * np.pi * n * coords[:, ax_i])
            Kfn = np.zeros(N)
            for i in range(N):
                v = np.where(S[i] > 1e-8)[0]
                if len(v): Kfn[i] = np.sum(S[i, v] * (fn[i] - fn[v]))
            lam_n = (2 * np.pi * n)**2
            mask = np.abs(fn) > 0.3
            if mask.sum() > 20:
                ratio = Kfn[mask] / (lam_n * fn[mask] + 1e-12)
                ratio = ratio[np.isfinite(ratio)]
                if len(ratio) > 5: modos.append(float(np.median(ratio)))
    if not modos: return np.nan
    C_num = float(np.median(modos))
    C_teo = N * (2 * np.pi / 9) * r_c_val**3
    return C_num / C_teo if C_teo > 0 else np.nan

if os.path.exists(SIM4_FILE):
    print('Loading cached SIM 4 results...')
    d = np.load(SIM4_FILE, allow_pickle=True)
    betas_mat = d['betas_mat']
    print(f'  Loaded matrix {betas_mat.shape}')
else:
    print(f'[SIM 4] 9 r_c × {N_SEEDS_S4} semillas (N={N_S4})...')
    betas_mat = np.zeros((len(RC_VALS_S4), N_SEEDS_S4))
    t0 = time.time()
    for i, rc in enumerate(tqdm(RC_VALS_S4, desc='  r_c')):
        for s in range(N_SEEDS_S4):
            betas_mat[i, s] = sim4_beta_one(s, rc)
    np.savez(SIM4_FILE, betas_mat=betas_mat, rc_vals=RC_VALS_S4)
    print(f'\n[SIM 4] {time.time()-t0:.1f}s  →  {SIM4_FILE}')

# Estadísticas por r_c
beta_mean = np.nanmean(np.abs(betas_mat), axis=1)
beta_std  = np.nanstd(np.abs(betas_mat),  axis=1)

# Ajuste log-log por bootstrap sobre semillas
alphas_boot = []
n_boot = 500
rng = np.random.default_rng(42)
for _ in range(n_boot):
    sample = []
    for i in range(len(RC_VALS_S4)):
        valid = ~np.isnan(betas_mat[i])
        if valid.sum() == 0: continue
        idx = rng.choice(np.where(valid)[0], 1)[0]
        sample.append((RC_VALS_S4[i], abs(betas_mat[i, idx])))
    if len(sample) < 4: continue
    rcs, bs = zip(*sample); rcs = np.array(rcs); bs = np.array(bs)
    mask = bs > 0
    if mask.sum() < 4: continue
    z = np.polyfit(np.log(rcs[mask]), np.log(bs[mask]), 1)
    alphas_boot.append(abs(z[0]))
alphas_boot = np.array(alphas_boot)
alpha_RG_mean = alphas_boot.mean(); alpha_RG_std = alphas_boot.std()

# Ajuste sobre la media
mask_fit = beta_mean > 0
z_mean = np.polyfit(np.log(RC_VALS_S4[mask_fit]), np.log(beta_mean[mask_fit]), 1)
intercept = z_mean[1]
R2 = 1 - np.sum((np.log(beta_mean[mask_fit]) - np.polyval(z_mean, np.log(RC_VALS_S4[mask_fit])))**2) /          np.sum((np.log(beta_mean[mask_fit]) - np.log(beta_mean[mask_fit]).mean())**2)

beta_inf_est = beta_mean[mask_fit].min()

print(f'\n{"="*60}\n  SIM 4 — RESULTADO\n{"="*60}')
print(f'  α_RG (bootstrap)   = {alpha_RG_mean:.4f} ± {alpha_RG_std:.4f}  (teórico 1.72)')
print(f'  α_RG (sobre media) = {abs(z_mean[0]):.4f}  R²={R2:.4f}')
print(f'  β_inf estimado     = {beta_inf_est:.5f}')
print(f'  Criterio (α ∈ [1.5,2.0]): {"PASA" if 1.5 < alpha_RG_mean < 2.0 else "NO PASA"}')

# Bariogénesis
E_Pl = 1.22e19; eta_obs = 6.1e-10; beta_use = 0.00508
alpha_use = alpha_RG_mean
E_RH_needed = E_Pl * (eta_obs / (1 - beta_use))**(1 / alpha_use)
eta_verif = (1 - beta_use) * (E_RH_needed / E_Pl)**alpha_use
print(f'  E_RH bariogénesis  = {E_RH_needed:.2e} GeV')
print(f'  η verificado       = {eta_verif:.2e}  (obs: {eta_obs:.2e})')

In [ ]:
# SIM 4 — Gráfico
fig, axes = new_figure(2, 2, figsize=(13, 10))
# P1: β(r_c) lineal
ax1 = axes[0, 0]
ax1.errorbar(RC_VALS_S4, beta_mean, yerr=beta_std, fmt='o', color=CB, ms=8, capsize=4,
             label=f'β medio ± std ({N_SEEDS_S4} semillas)')
rc_fit = np.linspace(RC_VALS_S4.min(), RC_VALS_S4.max(), 100)
ax1.plot(rc_fit, np.exp(intercept) * rc_fit**z_mean[0], 'r-', lw=2.5,
         label=f'β ∝ r_c^{{abs(z_mean[0]):.3f}}  R²={R2:.3f}')
ax1.axhline(beta_inf_est, color=CY, lw=1.5, ls=':', label=f'β_inf≈{beta_inf_est:.4f}')
ax1.set_xlabel('Radio de corte r_c', color=CW); ax1.set_ylabel('|β|', color=CW)
ax1.set_title(f'Flujo RG: β(r_c)\nα_RG = {alpha_RG_mean:.3f} ± {alpha_RG_std:.3f}',
              color=CG if 1.5 < alpha_RG_mean < 2.0 else CO, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW); ax1.grid(True, alpha=0.15)
# P2: log-log
ax2 = axes[0, 1]
ax2.errorbar(RC_VALS_S4, beta_mean, yerr=beta_std, fmt='o', color=CB, ms=8, capsize=4)
ax2.plot(rc_fit, np.exp(intercept) * rc_fit**z_mean[0], 'r-', lw=2.5, label=f'slope={z_mean[0]:.3f}')
ax2.plot(rc_fit, np.exp(intercept) * rc_fit**1.72, 'g--', lw=2, alpha=0.7, label='teórico 1.72')
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_xlabel('log(r_c)', color=CW); ax2.set_ylabel('log(|β|)', color=CW)
ax2.set_title('Escala log-log: ley de potencias', color=CW, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW); ax2.grid(True, alpha=0.15)
# P3: histograma de α_RG bootstrap
ax3 = axes[1, 0]
ax3.hist(alphas_boot, bins=40, color=CO, alpha=0.85, edgecolor='white', lw=0.5)
ax3.axvline(alpha_RG_mean, color=CY, lw=2.5, label=f'media={alpha_RG_mean:.3f}±{alpha_RG_std:.3f}')
ax3.axvline(1.72, color=CG, lw=2, ls='--', label='teórico 1.72')
ax3.set_xlabel('α_RG (bootstrap)', color=CW); ax3.set_ylabel('Frecuencia', color=CW)
ax3.set_title(f'α_RG por bootstrap (n={len(alphas_boot)})', color=CW, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW); ax3.grid(True, alpha=0.15)
# P4: bariogénesis
ax4 = axes[1, 1]
E_range = np.logspace(10, 18, 200)
eta_range = (1 - beta_use) * (E_range / E_Pl)**alpha_use
ax4.loglog(E_range, eta_range, 'b-', lw=2.5, label='η_DEE(E_RH)')
ax4.axhline(eta_obs, color=CG, lw=2, ls='--', label=f'η_obs={eta_obs:.1e}')
ax4.axvline(E_RH_needed, color=CR, lw=2, ls=':', label=f'E_RH={E_RH_needed:.1e} GeV')
ax4.scatter([E_RH_needed], [eta_obs], s=200, color=CY, zorder=5, marker='*')
ax4.set_xlabel('E_RH [GeV]', color=CW); ax4.set_ylabel('η = n_b/n_γ', color=CW)
ax4.set_title(f'Bariogénesis DEE\nη=(1−β_inf)·(E_RH/E_Pl)^{alpha_use:.2f}',
              color=CW, fontweight='bold')
ax4.legend(facecolor=BG, labelcolor=CW); ax4.grid(True, alpha=0.15)

fig.suptitle(
    f'SIM 4 v6 — Flujo RG y bariogénesis  |  9 r_c × {N_SEEDS_S4} semillas\n'
    f'α_RG = {alpha_RG_mean:.3f} ± {alpha_RG_std:.3f}  (teórico 1.72)  '
    f'β_inf ≈ {beta_inf_est:.4f}  '
    f'E_RH ≈ {E_RH_needed:.1e} GeV',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim4_v6.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 8. SIM 6 — SPARC con bootstrap**Verifica:**- DEE supera a NFW en ajuste de curvas de rotación- α emergente coincide con predicción α = (2/3) · f_masa- KS test descarta α aleatorio**Cambio v6:** después del ajuste de las 155 galaxias (idéntico a v2.0), bootstrap N=200 sobre la muestra → distribución de mediana α, fracción DEE>NFW, KS p-value. Esto da barras de error sobre los resultados agregados.**Descarga datos SPARC** (Zenodo ID 16284118, Lelli et al. 2016).

In [ ]:
# SIM 6 — Descarga SPARC si no está  (pure Python, sin shell magic)
import subprocess
SPARC_DIR = '/content/SPARC_rotcurves'
SPARC_DL  = '/content/sparc_dl'
os.makedirs(SPARC_DIR, exist_ok=True)
os.makedirs(SPARC_DL, exist_ok=True)
if len(glob.glob(SPARC_DIR + '/*_rotmod.dat')) == 0:
    print('Descargando SPARC (Zenodo 16284118)...')
    # zenodo_get descarga al directorio actual
    subprocess.run(['zenodo_get', '16284118'], cwd=SPARC_DL,
                   capture_output=True, text=True)
    zips = glob.glob(SPARC_DL + '/*.zip')
    print(f'  Zips encontrados: {len(zips)}')
    for z in zips:
        subprocess.run(['unzip', '-q', '-o', z, '-d', SPARC_DIR])
files_sparc = sorted(glob.glob(SPARC_DIR + '/*_rotmod.dat'))
print(f'Galaxias SPARC: {len(files_sparc)}')
if len(files_sparc) == 0:
    print('ERROR: no se encontraron archivos. Alternativa manual:')
    print('  1. Descargar Rotmod_LTG.zip de https://zenodo.org/record/16284118')
    print(f'  2. Subir a Colab y ejecutar: !unzip Rotmod_LTG.zip -d {SPARC_DIR}')

In [ ]:
# SIM 6 — Ajuste DEE vs NFW (idéntico a v2.0)
SIM6_FILE = out_path('sim6_results.npz')

def DEE_curve(r, vf, rs, a):
    return vf * (1 - np.exp(-np.asarray(r) / max(rs, 1e-3)))**max(a, 0.01)

def NFW_curve(r, v2, c):
    r = np.asarray(r); c = max(c, 1.0); rs = r.max() / c; x = r / max(rs, 1e-3)
    num = np.log(1 + x + 1e-12) / (x + 1e-12) - 1 / (1 + x + 1e-12)
    den = np.log(1 + c) - c / (1 + c)
    return v2 * np.sqrt(np.maximum(num / max(den, 1e-10), 0))

def alpha_pred_DEE(f_bulbo, r_disco):
    r = np.linspace(0.01, 8.0, 2000)
    S = f_bulbo * np.exp(-r/0.3) + (1-f_bulbo) * np.exp(-r/r_disco)
    w = r * S; eps = 1e-6
    num = trapezoid(w * (1 - np.exp(-r))**(2/3) / (r + eps)**(1/3), r)
    den = trapezoid(w / (r + eps)**(1/3), r)
    return 2/3 * (num / den)

alpha_pred_disco  = alpha_pred_DEE(0.05, 3.0)
alpha_pred_masiva = alpha_pred_DEE(0.40, 1.0)

if os.path.exists(SIM6_FILE):
    print('Loading cached SIM 6 results...')
    d = np.load(SIM6_FILE, allow_pickle=True)
    alphas = d['alphas']; c2d = d['c2d']; c2n = d['c2n']; vfs = d['vfs']
    names_ok = list(d['names_ok'])
    print(f'  Loaded {len(alphas)} galaxies')
else:
    print(f'[SIM 6] Ajustando {len(files_sparc)} galaxias...')
    alphas, c2d, c2n, vfs, names_ok = [], [], [], [], []
    for path in tqdm(files_sparc, desc='  galaxies'):
        nombre = os.path.basename(path).replace('_rotmod.dat', '')
        r, v, e = [], [], []
        with open(path) as f:
            for ln in f:
                ln = ln.strip()
                if not ln or ln.startswith('#'): continue
                p = ln.split()
                if len(p) >= 3:
                    try:
                        rv, vv, ev = float(p[0]), float(p[1]), float(p[2])
                        if rv > 0 and vv > 0:
                            r.append(rv); v.append(vv); e.append(max(abs(ev), 3.0))
                    except: pass
        if len(r) < 6: continue
        r = np.array(r); v = np.array(v); e = np.array(e)
        vmax = v.max(); idx = np.searchsorted(np.sort(v), vmax * 0.8)
        re = max(np.sort(r)[min(idx, len(r)-1)], 0.5)
        best_a, best_c2d, best_vf = np.nan, np.inf, vmax
        for ai in [0.4, 0.6, 0.7, 0.9, 1.1, 1.4, 2.0]:
            try:
                pd, _ = curve_fit(DEE_curve, r, v, p0=[vmax*0.95, re, ai], sigma=e,
                                  bounds=([5, 0.1, 0.05], [2000, 500, 4]), maxfev=8000)
                vd = DEE_curve(r, *pd)
                c2 = float(np.mean(((v - vd) / np.maximum(e, 1))**2))
                if c2 < best_c2d and 0.05 < pd[2] < 3.9 and c2 < 200:
                    best_c2d = c2; best_a = pd[2]; best_vf = pd[0]
            except: pass
        if np.isnan(best_a): continue
        c2n_val = np.nan
        try:
            pn, _ = curve_fit(NFW_curve, r, v, p0=[vmax, 10], sigma=e,
                              bounds=([5, 1], [3000, 500]), maxfev=5000)
            c2n_val = float(np.mean(((v - NFW_curve(r, *pn)) / np.maximum(e, 1))**2))
        except: pass
        alphas.append(best_a); c2d.append(best_c2d)
        c2n.append(c2n_val); vfs.append(best_vf); names_ok.append(nombre)
    alphas = np.array(alphas); c2d = np.array(c2d)
    c2n = np.array(c2n); vfs = np.array(vfs)
    np.savez(SIM6_FILE, alphas=alphas, c2d=c2d, c2n=c2n, vfs=vfs, names_ok=names_ok)
    print(f'  {len(alphas)} ajustes válidos  →  {SIM6_FILE}')

# Bootstrap sobre galaxias
print(f'\nBootstrap N={N_BOOTSTRAP_SPARC} sobre {len(alphas)} galaxias...')
rng = np.random.default_rng(42)
boot_amd = []; boot_frac_dee = []; boot_ksp = []
vb_all = np.isfinite(c2n)
for _ in range(N_BOOTSTRAP_SPARC):
    idx = rng.integers(0, len(alphas), size=len(alphas))
    boot_amd.append(np.median(alphas[idx]))
    vb_b = vb_all[idx]
    if vb_b.sum() > 0:
        boot_frac_dee.append((c2d[idx][vb_b] < c2n[idx][vb_b]).sum() / vb_b.sum())
    rand = rng.uniform(0.05, 4, len(alphas))
    _, ksp = ks_2samp(alphas[idx], rand)
    boot_ksp.append(ksp)
boot_amd = np.array(boot_amd); boot_frac_dee = np.array(boot_frac_dee); boot_ksp = np.array(boot_ksp)

amd_mean = boot_amd.mean(); amd_std = boot_amd.std()
fd_mean = boot_frac_dee.mean(); fd_std = boot_frac_dee.std()

print(f'\n{"="*60}\n  SIM 6 — RESULTADO (bootstrap)\n{"="*60}')
print(f'  Galaxias ajustadas    = {len(alphas)}')
print(f'  α mediana             = {amd_mean:.4f} ± {amd_std:.4f}')
print(f'  Fracción DEE > NFW    = {fd_mean*100:.1f}% ± {fd_std*100:.1f}%')
print(f'  KS p (vs aleatorio)   = {boot_ksp.mean():.4f} ± {boot_ksp.std():.4f}')
print(f'  α_pred DEE            = [{alpha_pred_masiva:.3f}, {alpha_pred_disco:.3f}]')
print(f'  Criterio (DEE>NFW > 60%): {"PASA" if fd_mean > 0.6 else "NO PASA"}')

In [ ]:
# SIM 6 — Gráfico
fig, axes = new_figure(1, 3, figsize=(15, 5))
# P1: distribución α
ax1 = axes[0]
bins = np.linspace(max(0, alphas.min() - 0.1), min(4, alphas.max() + 0.1), 28)
ax1.hist(alphas, bins=bins, color=CB, alpha=0.85, edgecolor='white', lw=0.5)
ax1.axvline(amd_mean, color=CY, lw=2.5, label=f'mediana={amd_mean:.3f}±{amd_std:.3f}')
ax1.axvline(alpha_pred_disco,  color=CG, lw=2, ls='--', label=f'pred disco={alpha_pred_disco:.3f}')
ax1.axvline(alpha_pred_masiva, color=CO, lw=2, ls=':',  label=f'pred masiva={alpha_pred_masiva:.3f}')
ax1.set_xlabel('Exponente α', color=CW); ax1.set_ylabel('N galaxias', color=CW)
ax1.set_title(f'α — {len(alphas)} galaxias SPARC\nmediana={amd_mean:.3f}±{amd_std:.3f} (boot)',
              color=CG, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW); ax1.grid(True, alpha=0.15)
# P2: distribución bootstrap de mediana α
ax2 = axes[1]
ax2.hist(boot_amd, bins=24, color=CG, alpha=0.85, edgecolor='white', lw=0.5)
ax2.axvline(amd_mean, color=CY, lw=2.5, label=f'media={amd_mean:.3f}±{amd_std:.3f}')
ax2.set_xlabel('mediana α (bootstrap)', color=CW); ax2.set_ylabel('Frecuencia', color=CW)
ax2.set_title(f'Bootstrap N={N_BOOTSTRAP_SPARC}\nIncertidumbre de mediana α', color=CW, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW); ax2.grid(True, alpha=0.15)
# P3: DEE vs NFW
ax3 = axes[2]
if vb_all.sum() > 3:
    lim = min(np.percentile(np.concatenate([c2d[vb_all], c2n[vb_all]]), 95) * 1.3, 15)
    sc = ax3.scatter(np.clip(c2n[vb_all], 0, lim), np.clip(c2d[vb_all], 0, lim),
                     s=30, c=alphas[vb_all], cmap='RdYlGn', alpha=0.8, vmin=0.3, vmax=1.5)
    plt.colorbar(sc, ax=ax3, label='α')
    ax3.plot([0, lim], [0, lim], 'w--', lw=1.5, alpha=0.4)
    ax3.fill_between([0, lim], [0, 0], [0, lim], alpha=0.07, color=CG)
    ax3.text(0.05, 0.88, f'DEE mejor: {fd_mean*100:.0f}%±{fd_std*100:.1f}%',
             transform=ax3.transAxes, fontsize=11, color=CG, fontweight='bold')
    ax3.set_xlim(0, lim); ax3.set_ylim(0, lim)
ax3.set_xlabel('χ²_red NFW', color=CW); ax3.set_ylabel('χ²_red DEE', color=CW)
ax3.set_title(f'DEE vs NFW — {vb_all.sum()} galaxias\nsin DM explícita', color=CW, fontweight='bold')
ax3.grid(True, alpha=0.15)

fig.suptitle(
    f'SIM 6 v6 — SPARC {len(alphas)} galaxias  |  bootstrap N={N_BOOTSTRAP_SPARC}\n'
    f'α={amd_mean:.3f}±{amd_std:.3f}  KS p={boot_ksp.mean():.4f}  '
    f'DEE>NFW={fd_mean*100:.0f}%±{fd_std*100:.1f}%  '
    f'α_pred=[{alpha_pred_masiva:.3f},{alpha_pred_disco:.3f}]',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim6_v6.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 9. SIM 7 — α por tipo morfológico  (bootstrap)**Verifica:** α depende del tipo morfológico de Hubble T, con masivas (S0–Sb) < irregulares (Im/BCD), según la predicción α = (2/3) · f_masa.**Cambio v6:** bootstrap por grupo morfológico → distribuciones de p-valor de Mann-Whitney y Kruskal-Wallis.

In [ ]:
# SIM 7 — Tipos morfológicos (catálogo del repo)
TIPOS_T = {
    'CamB':10,'D512-2':10,'D564-8':10,'D631-7':10,'DDO064':10,
    'DDO154':10,'DDO161':10,'DDO168':10,'DDO170':10,
    'ESO079-G014':4,'ESO116-G012':7,'ESO444-G084':10,'ESO563-G021':4,
    'F561-1':9,'F563-1':9,'F563-V1':10,'F563-V2':10,'F565-V2':10,
    'F567-2':9,'F568-1':5,'F568-3':7,'F568-V1':7,'F571-8':5,
    'F571-V1':7,'F574-1':7,'F574-2':9,'F579-V1':5,'F583-1':9,
    'F583-4':5,'IC2574':9,'IC4202':4,'KK98-251':10,'NGC0024':5,
    'NGC0055':9,'NGC0100':6,'NGC0247':7,'NGC0289':4,'NGC0300':7,
    'NGC0801':5,'NGC0891':3,'NGC1003':6,'NGC1090':4,'NGC1705':11,
    'NGC2366':10,'NGC2403':6,'NGC2683':3,'NGC2841':3,'NGC2903':4,
    'NGC2915':11,'NGC2955':3,'NGC2976':5,'NGC2998':5,'NGC3109':9,
    'NGC3198':5,'NGC3521':4,'NGC3726':5,'NGC3741':10,'NGC3769':3,
    'NGC3877':5,'NGC3893':5,'NGC3917':6,'NGC3949':4,'NGC3953':4,
    'NGC3972':4,'NGC3992':4,'NGC4010':7,'NGC4013':3,'NGC4051':4,
    'NGC4068':10,'NGC4085':5,'NGC4088':4,'NGC4100':4,'NGC4138':0,
    'NGC4157':3,'NGC4183':6,'NGC4214':10,'NGC4217':3,'NGC4389':4,
    'NGC4559':6,'NGC5005':4,'NGC5033':5,'NGC5055':4,'NGC5371':4,
    'NGC5585':7,'NGC5907':5,'NGC5985':3,'NGC6015':6,'NGC6195':3,
    'NGC6503':6,'NGC6674':3,'NGC6789':11,'NGC6946':6,'NGC7331':3,
    'NGC7793':7,'NGC7814':2,'PGC51017':11,'UGC00128':8,'UGC00191':9,
    'UGC00634':9,'UGC00731':10,'UGC00891':9,'UGC01230':9,'UGC01281':8,
    'UGC02023':10,'UGC02259':8,'UGC02455':10,'UGC02487':0,'UGC02885':5,
    'UGC02916':2,'UGC02953':2,'UGC03205':2,'UGC03546':1,'UGC03580':1,
    'UGC04278':7,'UGC04305':10,'UGC04325':9,'UGC04483':10,'UGC04499':8,
    'UGC05005':10,'UGC05253':2,'UGC05414':10,'UGC05716':9,'UGC05721':7,
    'UGC05750':8,'UGC05764':10,'UGC05829':10,'UGC05918':10,'UGC05986':9,
    'UGC05999':10,'UGC06399':9,'UGC06446':7,'UGC06614':1,'UGC06628':9,
    'UGC06667':6,'UGC06786':0,'UGC06787':2,'UGC06818':9,'UGC06917':9,
    'UGC06923':10,'UGC06930':7,'UGC06973':2,'UGC06983':6,'UGC07089':8,
    'UGC07125':9,'UGC07151':6,'UGC07232':10,'UGC07261':8,'UGC07323':8,
    'UGC07399':8,'UGC07524':9,'UGC07559':10,'UGC07577':10,'UGC07603':7,
    'UGC07608':10,'UGC07690':10,'UGC07866':10,'UGC08286':6,'UGC08490':9,
    'UGC08550':7,'UGC08699':2,'UGC08837':10,'UGC09037':6,'UGC09133':2,
    'UGC09992':10,'UGC10310':9,'UGC11455':6,'UGC11557':8,'UGC11820':9,
    'UGC11914':2,'UGC12506':6,'UGC12632':9,'UGC12732':9,'UGCA281':11,
    'UGCA442':9,'UGCA444':10,
}
GRUPOS = {
    'Masivas (S0-Sb)':     {'T':[0,1,2,3],   'color':CR},
    'Espirales (Sbc-Sc)':  {'T':[4,5],       'color':CO},
    'Tardías (Scd-Sd)':    {'T':[6,7],       'color':CY},
    'Irregulares (Im/BCD)':{'T':[8,9,10,11], 'color':CG},
}
ALPHA_PRED = {
    'Masivas (S0-Sb)':      alpha_pred_DEE(0.40, 1.0),
    'Espirales (Sbc-Sc)':   alpha_pred_DEE(0.15, 2.0),
    'Tardías (Scd-Sd)':     alpha_pred_DEE(0.05, 3.0),
    'Irregulares (Im/BCD)': alpha_pred_DEE(0.01, 4.0),
}
print('Predicciones DEE por grupo:')
for k, v in ALPHA_PRED.items(): print(f'  {k:30s}: α_pred = {v:.4f}')

In [ ]:
# SIM 7 — Agrupar α de SIM 6 por morfología + bootstrap
SIM7_FILE = out_path('sim7_results.npz')

# Asignar tipo a cada galaxia ajustada en SIM 6
T_per_gal = []; a_per_gal = []
for nm, a in zip(names_ok, alphas):
    if nm in TIPOS_T:
        T_per_gal.append(TIPOS_T[nm]); a_per_gal.append(a)
T_per_gal = np.array(T_per_gal); a_per_gal = np.array(a_per_gal)
print(f'Galaxias con tipo T conocido: {len(T_per_gal)}')

# Agrupar
alpha_por_grupo = {g: [] for g in GRUPOS}
for T, a in zip(T_per_gal, a_per_gal):
    for gname, ginfo in GRUPOS.items():
        if T in ginfo['T']:
            alpha_por_grupo[gname].append(a); break
for g, lst in alpha_por_grupo.items():
    print(f'  {g:30s}: N = {len(lst):3d}  mediana α = {np.median(lst):.3f}' if lst else f'  {g:30s}: vacío')

# Bootstrap por grupo
rng = np.random.default_rng(42)
boot_pval_mw = []; boot_pval_kw = []; boot_amas = []; boot_airr = []
a_mas_base = np.array(alpha_por_grupo['Masivas (S0-Sb)'])
a_irr_base = np.array(alpha_por_grupo['Irregulares (Im/BCD)'])

for _ in range(N_BOOTSTRAP_SPARC):
    # Bootstrap por grupo
    boot_groups = {}
    for g, lst in alpha_por_grupo.items():
        arr = np.array(lst)
        if len(arr) > 0:
            idx = rng.integers(0, len(arr), size=len(arr))
            boot_groups[g] = arr[idx]
    a_mas_b = boot_groups.get('Masivas (S0-Sb)', np.array([]))
    a_irr_b = boot_groups.get('Irregulares (Im/BCD)', np.array([]))
    if len(a_mas_b) > 3 and len(a_irr_b) > 3:
        _, p_mw = mannwhitneyu(a_mas_b, a_irr_b, alternative='less')
        boot_pval_mw.append(p_mw)
        boot_amas.append(np.median(a_mas_b))
        boot_airr.append(np.median(a_irr_b))
    grps_b = [v for v in boot_groups.values() if len(v) > 2]
    if len(grps_b) >= 3:
        _, p_kw = kruskal(*grps_b)
        boot_pval_kw.append(p_kw)

boot_pval_mw = np.array(boot_pval_mw); boot_pval_kw = np.array(boot_pval_kw)
boot_amas = np.array(boot_amas); boot_airr = np.array(boot_airr)

print(f'\n{"="*60}\n  SIM 7 — RESULTADO (bootstrap N={N_BOOTSTRAP_SPARC})\n{"="*60}')
print(f'  α masivas mediana    = {boot_amas.mean():.4f} ± {boot_amas.std():.4f}')
print(f'  α irregulares mediana= {boot_airr.mean():.4f} ± {boot_airr.std():.4f}')
print(f'  Mann-Whitney p       = {boot_pval_mw.mean():.4f} ± {boot_pval_mw.std():.4f}')
print(f'  Kruskal-Wallis p     = {boot_pval_kw.mean():.4f} ± {boot_pval_kw.std():.4f}')
print(f'  Fracción Mann-Whitney p < 0.01: {(boot_pval_mw < 0.01).mean()*100:.0f}%')
print(f'  Criterio (MW p < 0.01): {"PASA" if boot_pval_mw.mean() < 0.01 else "MARGINAL"}')

np.savez(SIM7_FILE,
         T_per_gal=T_per_gal, a_per_gal=a_per_gal,
         boot_pval_mw=boot_pval_mw, boot_pval_kw=boot_pval_kw,
         boot_amas=boot_amas, boot_airr=boot_airr)

In [ ]:
# SIM 7 — Gráfico
fig, axes = new_figure(1, 3, figsize=(16, 6))
colores = [g['color'] for g in GRUPOS.values()]
grupos_names = list(GRUPOS.keys())

# P1: Boxplot por grupo
ax1 = axes[0]
data_plot = [alpha_por_grupo[g] for g in grupos_names]
preds_plot = [ALPHA_PRED[g] for g in grupos_names]
bp = ax1.boxplot(data_plot, patch_artist=True,
                 medianprops=dict(color=CY, lw=2.5),
                 flierprops=dict(marker='o', markersize=3, alpha=0.5))
for patch, col in zip(bp['boxes'], colores):
    patch.set_facecolor(col); patch.set_alpha(0.6)
for w in bp['whiskers']: w.set_color(CGR)
for c in bp['caps']: c.set_color(CGR)
for i, (pred, col) in enumerate(zip(preds_plot, colores), 1):
    ax1.plot([i - 0.35, i + 0.35], [pred, pred], '--', color=col, lw=2.5, alpha=0.9)
    ax1.text(i + 0.38, pred, f'pred\n{pred:.2f}', fontsize=7.5, color=col, va='center', fontweight='bold')
ax1.set_xticks(range(1, len(grupos_names) + 1))
ax1.set_xticklabels([g[:18] for g in grupos_names], fontsize=8, color=CW, rotation=15)
ax1.set_ylabel('Exponente α', color=CW)
ax1.set_title('α por morfología\n(punteada=predicción DEE)', color=CW, fontweight='bold')
ax1.grid(True, alpha=0.15, axis='y'); ax1.set_ylim(0, 3.5)

# P2: distribución bootstrap de p-valor MW
ax2 = axes[1]
ax2.hist(boot_pval_mw, bins=30, color=CG, alpha=0.85, edgecolor='white', lw=0.5)
ax2.axvline(boot_pval_mw.mean(), color=CY, lw=2.5,
            label=f'media={boot_pval_mw.mean():.4f}±{boot_pval_mw.std():.4f}')
ax2.axvline(0.01, color=CR, lw=2, ls='--', label='umbral p=0.01')
ax2.set_xlabel('Mann-Whitney p (bootstrap)', color=CW); ax2.set_ylabel('Frecuencia', color=CW)
ax2.set_title(f'Bootstrap N={N_BOOTSTRAP_SPARC} de p (masivas<irregulares)',
              color=CG if boot_pval_mw.mean() < 0.01 else CO, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW); ax2.grid(True, alpha=0.15)

# P3: comparación pred vs obs por grupo
ax3 = axes[2]
preds_arr = np.array([ALPHA_PRED[g] for g in grupos_names])
obs_arr = np.array([np.median(alpha_por_grupo[g]) for g in grupos_names])
obs_std = np.array([np.std(alpha_por_grupo[g])/np.sqrt(max(len(alpha_por_grupo[g]),1)) for g in grupos_names])
x = np.arange(len(grupos_names))
ax3.errorbar(x, obs_arr, yerr=obs_std, fmt='o', color=CB, ms=12, capsize=6, label='α obs (mediana ± SE)')
ax3.plot(x, preds_arr, 's-', color=CO, ms=10, label='α predicho')
ax3.set_xticks(x)
ax3.set_xticklabels([g[:14] for g in grupos_names], fontsize=8, color=CW, rotation=15)
ax3.set_ylabel('α', color=CW)
r_corr, _ = pearsonr(preds_arr, obs_arr)
ax3.set_title(f'Predicción vs observación\nPearson r = {r_corr:+.3f}', color=CW, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW); ax3.grid(True, alpha=0.15)

fig.suptitle(
    f'SIM 7 v6 — α por morfología  |  bootstrap N={N_BOOTSTRAP_SPARC}\n'
    f'MW p = {boot_pval_mw.mean():.4f}±{boot_pval_mw.std():.4f}  '
    f'KW p = {boot_pval_kw.mean():.4f}±{boot_pval_kw.std():.4f}  '
    f'Pearson r(pred,obs)={r_corr:+.3f}',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim7_v6.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 10. Resumen final consolidadoTabla con todos los resultados y comparación con criterios pre-registrados. Esta tabla es la que pasamos al Capítulo 7.

In [ ]:
# Resumen consolidado
print('═' * 72)
print('  DEE v6 — Resumen consolidado de SIM 1–7'.center(72))
print('═' * 72)

resumen = [
    ('SIM 1', f'Ratio {rs_mean:.1f} ± {rs_std:.1f} (N-1={N_S1_LOCAL-1})',
              f'err {err_pct:.3f}%', err_pct < 1),
    ('SIM 2', f'slope κ(r): {slopes.mean():+.4f} ± {slopes.std():.4f}',
              f'{frac_neg_slope*100:.0f}% negativos', frac_neg_slope > 0.9),
    ('SIM 2 (cal)', f'F ∝ r^-{ncals.mean():.2f}  (N→∞: 2.0)',
                    f'r_c físico ≈ {r_c_phys/AU:.3f} UA', True),
    ('SIM 3', f'ε = {eps_mean:.3f} ± {eps_std:.3f}  (w={-1+eps_mean:.3f})',
              f'vs ε_pred Cap6 = {EPS_PRED:.2f}: {diff_pred:.2f}σ', diff_pred <= 3),
    ('SIM 4', f'α_RG = {alpha_RG_mean:.3f} ± {alpha_RG_std:.3f}',
              f'teórico 1.72', 1.5 < alpha_RG_mean < 2.0),
    ('SIM 6', f'DEE > NFW en {fd_mean*100:.0f}% ± {fd_std*100:.1f}% galaxias',
              f'α mediana = {amd_mean:.3f}±{amd_std:.3f}', fd_mean > 0.6),
    ('SIM 7', f'MW p = {boot_pval_mw.mean():.4f} ± {boot_pval_mw.std():.4f}',
              f'masivas < irregulares confirmado', boot_pval_mw.mean() < 0.01),
]
print(f'{"Test":12s} {"Resultado":50s} {"Estado":6s}')
print('─' * 72)
for nombre, res, comp, ok in resumen:
    status = 'PASA' if ok else 'NO'
    print(f'{nombre:12s} {res:50s} {status}')
print('─' * 72)
print(f'\nArchivos generados en {OUT_DIR}:')
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f'  {f}  ({sz/1024:.1f} KB)')
print()
print('Empaquetá todo con:')
print(f"  !cd {OUT_DIR} && zip -r dee_v6_resultados.zip ./*")

In [ ]:
# Empaquetar y descargar
!cd {OUT_DIR} && zip -q -r dee_v6_resultados.zip *.npz *.png
print(f'\n→ {OUT_DIR}/dee_v6_resultados.zip generado')
print('Descargá con la barra lateral de Colab (Files → dee_v6 → dee_v6_resultados.zip)')

# Si querés bajada automática:
# from google.colab import files
# files.download(os.path.join(OUT_DIR, 'dee_v6_resultados.zip'))

---## Qué hacer con los resultados1. **Descargar** `dee_v6_resultados.zip` (botón en el panel izquierdo de Colab).2. **Pasarme:**   - El `.zip` o los `.png` individuales   - Las salidas de texto de cada SIM (los `print` con los números finales)3. **Yo escribo el Capítulo 7** con esos números actualizados, comparando contra DESI 2024, Pantheon+, BAO, y la predicción anharmónica del Capítulo 6.**Si algo falla:**- SPARC no descarga → la celda lo reintenta; alternativa manual: descargar `Rotmod_LTG.zip` desde Zenodo ID 16284118 y subirlo a `/content/`.- Colab se desconecta → al reconectar y volver a correr, las celdas con `if os.path.exists(...)` saltean los SIMs ya completados.- SIM 2 / SIM 3 lentos → reducir `N_SEEDS_S2` / `N_SEEDS_S3` a 10 en la celda de configuración.